In [17]:
!unzip archive.zip -d dataset



Archive:  archive.zip
replace dataset/MSR-LA - 3467.docx? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace dataset/PetImages/Cat/0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: [n]o
error:  invalid response [[n]o]
replace dataset/PetImages/Cat/0.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace dataset/PetImages/Cat/1.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import itertools
import copy
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
n

Device: cuda


In [19]:
def clean_petimages(root):

    for cls in ["Cat", "Dog"]:
        folder = os.path.join(root, cls)

        for f in os.listdir(folder):
            path = os.path.join(folder, f)
            try:
                img = Image.open(path)
                img.verify()
            except:
                os.remove(path)

In [20]:
def get_cats_dogs_loaders(root="./dataset/PetImages",
                          batch_size=32,
                          train_ratio=0.8):

    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor()
    ])

    dataset = datasets.ImageFolder(root, transform=transform)

    train_size = int(train_ratio * len(dataset))
    val_size = len(dataset) - train_size

    train_ds, val_ds = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=2)

    val_loader = DataLoader(val_ds, batch_size=batch_size,
                            shuffle=False, num_workers=2)

    return train_loader, val_loader

In [21]:
def get_cifar10_loaders(batch_size=64):

    transform = transforms.Compose([
        transforms.ToTensor()
    ])

    train_ds = datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform)

    test_ds = datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=2)

    test_loader = DataLoader(test_ds, batch_size=batch_size,
                              shuffle=False, num_workers=2)

    return train_loader, test_loader

In [22]:
class CNN(nn.Module):

    def __init__(self, num_classes=2, activation="relu"):
        super().__init__()

        if activation == "relu":
            act = nn.ReLU(inplace=True)
        elif activation == "tanh":
            act = nn.Tanh()
        elif activation == "leakyrelu":
            act = nn.LeakyReLU(0.1, inplace=True)

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            act,
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            act,
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            act,
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            act,
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [23]:
def initialize_weights(model, init_type="xavier"):

    for m in model.modules():

        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):

            if init_type == "xavier":
                nn.init.xavier_uniform_(m.weight)

            elif init_type == "kaiming":
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")

            elif init_type == "random":
                nn.init.normal_(m.weight, mean=0.0, std=0.02)

            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

In [27]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

activations = ["relu", "tanh", "leakyrelu"]
inits = ["xavier", "kaiming", "random"]
optimizers = ["sgd", "adam", "rmsprop"]


def train_one_setting(train_loader, val_loader,
                      activation, init_type, optimizer_name,
                      num_classes, epochs=5):

    model = CNN(num_classes=num_classes,
                activation=activation).to(device)

    initialize_weights(model, init_type)

    if optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    elif optimizer_name == "adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    elif optimizer_name == "rmsprop":
        optimizer = torch.optim.RMSprop(model.parameters(), lr=0.001)

    criterion = nn.CrossEntropyLoss()

    print(f"\nConfig: {activation} | {init_type} | {optimizer_name}")

    best_acc = 0

    for epoch in range(epochs):


        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()


        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                pred = out.argmax(1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        val_acc = correct / total

        print(f"Epoch {epoch+1}: Val Acc={val_acc:.4f}")

        best_acc = max(best_acc, val_acc)

    return best_acc


def run_cats_dogs():

    print("Cleaning PetImages dataset...")
    clean_petimages("./dataset/PetImages")

    train_loader, val_loader = get_cats_dogs_loaders(
        "./dataset/PetImages"
    )

    os.makedirs("saved_models", exist_ok=True)

    results = []

    for a in activations:
        for i in inits:
            for o in optimizers:
                name = f"catsdogs_{a}_{i}_{o}"
                acc = train_one_setting(
                    train_loader, val_loader,
                    a, i, o,
                    num_classes=2
                )
                results.append((name, acc))

    print("\nCats vs Dogs results")
    for r in results:
        print(r)


def run_cifar10():

    train_loader, val_loader = get_cifar10_loaders()

    os.makedirs("saved_models", exist_ok=True)

    results = []

    for a in activations:
        for i in inits:
            for o in optimizers:
                name = f"cifar10_{a}_{i}_{o}"
                acc = train_one_setting(
                    train_loader, val_loader,
                    a, i, o,
                    num_classes=10
                )
                results.append((name, acc))

    print("\nCIFAR-10 results")
    for r in results:
        print(r)


if __name__ == "__main__":

    run_cats_dogs()
    run_cifar10()

Cleaning PetImages dataset...


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))



Config: relu | xavier | sgd
Epoch 1: Val Acc=0.6056
Epoch 2: Val Acc=0.7028
Epoch 3: Val Acc=0.6656
Epoch 4: Val Acc=0.6882
Epoch 5: Val Acc=0.7466

Config: relu | xavier | adam
Epoch 1: Val Acc=0.7260
Epoch 2: Val Acc=0.7636
Epoch 3: Val Acc=0.6720
Epoch 4: Val Acc=0.7166
Epoch 5: Val Acc=0.7912

Config: relu | xavier | rmsprop
Epoch 1: Val Acc=0.7008
Epoch 2: Val Acc=0.7096
Epoch 3: Val Acc=0.7832
Epoch 4: Val Acc=0.8032
Epoch 5: Val Acc=0.8118

Config: relu | kaiming | sgd
Epoch 1: Val Acc=0.6710
Epoch 2: Val Acc=0.7084
Epoch 3: Val Acc=0.5978
Epoch 4: Val Acc=0.7178
Epoch 5: Val Acc=0.6648

Config: relu | kaiming | adam
Epoch 1: Val Acc=0.6000
Epoch 2: Val Acc=0.7548
Epoch 3: Val Acc=0.7266
Epoch 4: Val Acc=0.7966
Epoch 5: Val Acc=0.8100

Config: relu | kaiming | rmsprop
Epoch 1: Val Acc=0.6468
Epoch 2: Val Acc=0.6864
Epoch 3: Val Acc=0.7756
Epoch 4: Val Acc=0.7682
Epoch 5: Val Acc=0.7928

Config: relu | random | sgd
Epoch 1: Val Acc=0.6654
Epoch 2: Val Acc=0.6740
Epoch 3: Val Acc

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 2: Val Acc=0.7156


...

Epoch 4: Val Acc=0.7462


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 5: Val Acc=0.8060

Cats vs Dogs results
('catsdogs_relu_xavier_sgd', 0.7466)
('catsdogs_relu_xavier_adam', 0.7912)
('catsdogs_relu_xavier_rmsprop', 0.8118)
('catsdogs_relu_kaiming_sgd', 0.7178)
('catsdogs_relu_kaiming_adam', 0.81)
('catsdogs_relu_kaiming_rmsprop', 0.7928)
('catsdogs_relu_random_sgd', 0.7614)
('catsdogs_relu_random_adam', 0.7992)
('catsdogs_relu_random_rmsprop', 0.7644)
('catsdogs_tanh_xavier_sgd', 0.7162)
('catsdogs_tanh_xavier_adam', 0.774)
('catsdogs_tanh_xavier_rmsprop', 0.7678)
('catsdogs_tanh_kaiming_sgd', 0.722)
('catsdogs_tanh_kaiming_adam', 0.7788)
('catsdogs_tanh_kaiming_rmsprop', 0.766)
('catsdogs_tanh_random_sgd', 0.7008)
('catsdogs_tanh_random_adam', 0.7726)
('catsdogs_tanh_random_rmsprop', 0.7446)
('catsdogs_leakyrelu_xavier_sgd', 0.776)
('catsdogs_leakyrelu_xavier_adam', 0.8152)
('catsdogs_leakyrelu_xavier_rmsprop', 0.8168)
('catsdogs_leakyrelu_kaiming_sgd', 0.76)
('catsdogs_leakyrelu_kaiming_adam', 0.8026)
('catsdogs_leakyrelu_kaiming_rmsprop', 0.8

100%|██████████| 170M/170M [00:13<00:00, 12.7MB/s]



Config: relu | xavier | sgd
Epoch 1: Val Acc=0.4499
Epoch 2: Val Acc=0.6202
Epoch 3: Val Acc=0.6180
Epoch 4: Val Acc=0.6748
Epoch 5: Val Acc=0.6792

Config: relu | xavier | adam
Epoch 1: Val Acc=0.4574
Epoch 2: Val Acc=0.6068
Epoch 3: Val Acc=0.6500
Epoch 4: Val Acc=0.6522
Epoch 5: Val Acc=0.6673

Config: relu | xavier | rmsprop
Epoch 1: Val Acc=0.4538
Epoch 2: Val Acc=0.4823
Epoch 3: Val Acc=0.4675
Epoch 4: Val Acc=0.5530
Epoch 5: Val Acc=0.5386

Config: relu | kaiming | sgd
Epoch 1: Val Acc=0.5182
Epoch 2: Val Acc=0.5216
Epoch 3: Val Acc=0.6387
Epoch 4: Val Acc=0.6584
Epoch 5: Val Acc=0.6475

Config: relu | kaiming | adam
Epoch 1: Val Acc=0.5397
Epoch 2: Val Acc=0.5667
Epoch 3: Val Acc=0.6487
Epoch 4: Val Acc=0.6536
Epoch 5: Val Acc=0.7050

Config: relu | kaiming | rmsprop
Epoch 1: Val Acc=0.5051
Epoch 2: Val Acc=0.5553
Epoch 3: Val Acc=0.5763
Epoch 4: Val Acc=0.4499
Epoch 5: Val Acc=0.6778

Config: relu | random | sgd
Epoch 1: Val Acc=0.5329
Epoch 2: Val Acc=0.5518
Epoch 3: Val Acc